# 编译与内存越权执行指南：基于 C 语言的底层探索

本指南基于 `shellcode_hnu.c` 的源码逻辑，详细拆解如何通过操作系统底层 API 绕过 NX（数据不可执行）保护，实现“数据即指令”的冯·诺依曼架构特性演示。

## 1. 理论背景：冯·诺依曼架构的盲区与 NX 保护
在经典的冯·诺依曼体系结构中，程序的指令和数据共享同一片物理存储空间（内存）。对于 CPU 而言，只要程序计数器（RIP 寄存器）指向某个内存地址，CPU 就会尝试将该处的数据作为机器指令进行取指、译码和执行。

为了防止恶意数据被当成代码执行，现代操作系统引入了 **NX Bit（No-eXecute，数据不可执行）** 硬件保护机制。存放常规变量（如我们的 `unsigned char` 数组）的数据段会被内核标记为不可执行，强行跳转会触发段错误（Segmentation fault）。本指南旨在通过合法 API 绕过此限制，展示数据向指令的跃迁。

## 2. 核心突破口：获取页大小与内存页对齐
修改内存权限时，系统 API 要求传入的地址必须是**内存页的绝对物理起始边界**。
```c
size_t page_size = sysconf(_SC_PAGESIZE);
uintptr_t page_start = (uintptr_t)shellcode & ~(page_size - 1);
```
**原理解析：**
1. 64 位 Linux 默认页大小为 4096 字节（4KB）。
2. `page_size - 1` 的值为 4095，其二进制低 12 位全为 1（`0xFFF`）。
3. 取反 `~` 后，得到一个高位全为 1、低 12 位全为 0 的掩码。
4. 将数组的内存地址与该掩码进行**按位与（`&`）** 操作，直接将地址的低 12 位强行“抹零”，精准计算出包含该数组的内存页的绝对起始物理地址。

## 3. 权限篡改：mprotect 系统调用
定位到目标内存页后，向内核发起越权请求：
```c
if (mprotect((void *)page_start, page_size, PROT_READ | PROT_WRITE | PROT_EXEC) == -1) {
    perror("[-] mprotect failed");
    return 1;
}
```
**原理解析：**
调用 `<sys/mman.h>` 库中的 `mprotect`，将该内存页的权限强行重写为 `PROT_READ`（可读）、`PROT_WRITE`（可写）和 **`PROT_EXEC`（可执行）**。执行成功后，NX Bit 的封印被彻底解除，该区域获得了合法执行的系统身份。

## 4. 指令跃迁：强转函数指针并执行
接管 CPU 的指令流，完成最终击杀：
```c
void (*execute_shellcode)() = (void (*)())shellcode;
execute_shellcode();
```
**原理解析：**
利用 C 语言的强制类型转换，将原本普通的数组指针强行赋予了“函数入口”的语义。当调用 `execute_shellcode()` 时，底层生成 `call` 指令，CPU 的 RIP 寄存器跳转至数组起始位置，将内部的十六进制数据（机器码）当做系统指令流畅执行，并安全返回。

## 5. 演示环境与操作规范
* **宿主机环境**：Ubuntu Linux x86_64
* **演示前端**：JupyterLab / RISE 交互式幻灯片
* **编译与执行步骤**：
  在 Jupyter Cell 中调用系统工具链：
  ```bash
  !gcc shellcode_hnu.c -o shellcode_hnu
  !./shellcode_hnu
  ```

## 6. 参考文献
1. **Linux Programmer's Manual (man pages)**
   * `mprotect(2)`: 深入解析 Linux 内存页访问权限控制机制及 PROT_EXEC 宏。
   * `sysconf(3)`: 解析 POSIX 操作系统在运行时获取系统配置信息（如 _SC_PAGESIZE）的标准接口。
2. **Intel® 64 and IA-32 Architectures Software Developer’s Manual**
   * *Volume 3: System Programming Guide*. 详细阐述 x86_64 架构下的分页机制（Paging）以及 Execute-Disable (XD) Bit 的硬件实现原理。
3. **System V Application Binary Interface (AMD64 Architecture Processor Supplement)**
   * Linux x86_64 系统的标准调用约定。规定了系统调用的触发方式及寄存器传参规则，为构建 Shellcode 提供了规范。
4. **Bryant, R. E., & O'Hallaron, D. R. (2015). *Computer Systems: A Programmer's Perspective* (3rd ed.). Pearson.**
   * 详述指令与数据的物理等价性及操作系统虚拟内存管理机制。

In [ ]:
%%writefile shellcode_hnu.c

In [ ]:
#include <stdio.h>
#include <sys/mman.h>   // 包含 mprotect
#include <unistd.h>     // 包含 sysconf
#include <stdint.h>     // 包含 uintptr_t

In [ ]:
unsigned char shellcode[] = {
    0x48, 0xc7, 0xc0, 0x01, 0x00, 0x00, 0x00, // mov rax, 1 (sys_write)
    0x48, 0xc7, 0xc7, 0x01, 0x00, 0x00, 0x00, // mov rdi, 1 (stdout)
    0x48, 0x8d, 0x35, 0x0a, 0x00, 0x00, 0x00, // lea rsi, [rip+0x0a] (指向下方字符串)
    0x48, 0xc7, 0xc2, 0x19, 0x00, 0x00, 0x00, // mov rdx, 25 (长度)
    0x0f, 0x05,                               // syscall (触发调用)
    0xc3,                                     // ret (返回)
    
    // "I Love Hunan University!\n" 的十六进制 ASCII 码
    0x49, 0x20, 0x4c, 0x6f, 0x76, 0x65, 0x20, 0x48,
    0x75, 0x6e, 0x61, 0x6e, 0x20, 0x55, 0x6e, 0x69,
    0x76, 0x65, 0x72, 0x73, 0x69, 0x74, 0x79, 0x21, 0x0a
};

### 1. 设置“我要做什么” (系统调用号)
```c
0x48, 0xc7, 0xc0, 0x01, 0x00, 0x00, 0x00  // mov rax, 1
```
* **汇编指令**：`mov rax, 1`
* **CPU 动作**：将 64 位寄存器 `rax` 的值设为 1。
* **原理**：在 Linux x86_64 架构中，`rax` 寄存器决定了你要调用的“服务编号”。编号 `1` 代表 **`sys_write`**（写操作）。这告诉内核：“准备好，我要往外写东西了。”

### 2. 设置“往哪儿写” (文件描述符)
```c
0x48, 0xc7, 0xc7, 0x01, 0x00, 0x00, 0x00  // mov rdi, 1
```
* **汇编指令**：`mov rdi, 1`
* **CPU 动作**：将 `rdi` 寄存器设为 1。
* **原理**：`rdi` 是系统调用的第一个参数。在 Linux 里，文件描述符 `1` 永远代表 **`stdout`（标准输出/屏幕）**。这告诉内核：“把我要写的内容显示在屏幕上。”

### 3. 设置“写什么” (数据的内存地址)
```c
0x48, 0x8d, 0x35, 0x0a, 0x00, 0x00, 0x00  // lea rsi, [rip+0x0a]
```
* **汇编指令**：`lea rsi, [rip + 10]`
* **CPU 动作**：**最天才的一步**。它计算当前指令指针（`rip`）往后偏移 10 个字节的地址，存入 `rsi`。
* **原理**：`rsi` 是第二个参数，存放要打印的字符串地址。因为我们的字符串就在这段机器码的末尾，所以通过“相对寻址”，代码无论被放在内存哪个角落，都能精准找到紧跟其后的 "I Love..."。

### 4. 设置“写多长”并扣动扳机
```c
0x48, 0xc7, 0xc2, 0x19, 0x00, 0x00, 0x00  // mov rdx, 25
0x0f, 0x05                                // syscall
```
* **汇编指令**：`mov rdx, 25` 和 `syscall`
* **CPU 动作**：将 `rdx`（第三个参数）设为 25（字符串字节长度），然后执行 `syscall`。
* **原理**：`syscall` 是一条特殊的指令，它会让 CPU 从“用户态”陷入“内核态”。内核看到 `rax=1`, `rdi=1`, `rsi=地址`, `rdx=25`，就会心领神会，直接在屏幕上喷出你的校名。



---

### 5. 安全撤离 (Return)
```c
0xc3                                      // ret
```
* **汇编指令**：`ret`
* **CPU 动作**：从栈中弹出返回地址，跳回 `main` 函数。
* **原理**：如果没有这一行，CPU 执行完写操作后会继续往后“乱跑”，尝试把内存里乱七八糟的数据当指令运行，结果就是死机。有了 `ret`，你的 C 程序就能在打印完校名后，优雅地接着跑后面的 `printf`。

In [ ]:
int main() {
    printf("[*] System Hacker Mode Activated\n");
    printf("[*] Shellcode address: %p\n", (void*)shellcode); 

```c
printf("[*] System Hacker Mode Activated\n");
```
* **讲解：** 这一行是纯粹的“仪式感”。它告诉用户程序已启动，并使用了 `[*]` 这个标准的安全工具输出格式，建立专业感。

```c
printf("[*] Shellcode address: %p\n", (void*)shellcode);
```
* **讲解：** 这是**关键证据一**。我们把数组在内存里的原始地址打印出来。你可以指着它说：“看，这是数据段的一个地址，正常情况下它是没有执行权限的。”

In [ ]:
    // 1. 获取系统内存页大小
    size_t page_size = sysconf(_SC_PAGESIZE);

    // 2. 计算页起始地址
    uintptr_t page_start = (uintptr_t)shellcode & ~(page_size - 1);
    printf("[*] Page start address: 0x%lx\n", page_start);

```c
size_t page_size = sysconf(_SC_PAGESIZE);
```
* **讲解：** **向内核打听规矩**。操作系统管理权限不是按字节，而是按“页”（通常 4KB）。这行代码获取当前系统的页大小，为下一步做准备。

```c
uintptr_t page_start = (uintptr_t)shellcode & ~(page_size - 1);
```
* **讲解：** **全场最硬核的位运算**。
    * `mprotect` 要求地址必须是页的开头（即 4096 的倍数）。
    * 通过 `& ~(4095)`，我们将地址的低位清零。
    * **演示亮点：** 如果打印出的 `shellcode` 地址是 `0x555555558040`，那么 `page_start` 就会变成 `0x555555558000`。

```c
printf("[*] Page start address: 0x%lx\n", page_start);
```
* **讲解：** **关键证据二**。打印对齐后的地址。你可以让台下看：**最后三位是不是 000？** 如果是，说明对齐成功！

In [ ]:
    // 3. 调用 mprotect，修改内存页权限为 可读-可写-可执行 (R-W-X)
    if (mprotect((void *)page_start, page_size, PROT_READ | PROT_WRITE | PROT_EXEC) == -1) {
        perror("[-] mprotect failed");
        return 1;
    }
    printf("[+] NX Bit bypassed successfully using mprotect!\n");
    printf("[+] Executing data array as machine instructions...\n\n");

```c
if (mprotect((void *)page_start, page_size, PROT_READ | PROT_WRITE | PROT_EXEC) == -1) {
```
* **讲解：** **暴力破门**。这一行直接呼叫 Linux 内核：“嘿，把这一页内存的锁给我拆了，我要读、要写、还要执行！”
    * `PROT_EXEC` 是核心，没有它，这块内存就是死的。

```c
perror("[-] mprotect failed");
```
* **讲解：** **安全气囊**。如果系统拒绝了你的越权请求（比如权限不足），`perror` 会打印出具体的错误原因，防止程序无声无息地崩溃。

```c
printf("[+] NX Bit bypassed successfully using mprotect!\n");
printf("[+] Executing data array as machine instructions...\n\n");
```
* **讲解：** **胜利宣言**。这两行 `printf` 是在告诉观众：我们已经成功绕过了系统的 **NX（不可执行）保护**，接下来就是见证奇迹的时刻。


In [7]:
    // 4. 强转函数指针并执行！
    void (*execute_shellcode)() = (void (*)())shellcode;
    execute_shellcode();

    printf("\n[+] Execution finished safely.\n");
    return 0;
}

Overwriting shellcode_hnu.c


```c
void (*execute_shellcode)() = (void (*)())shellcode;
```
* **讲解：** **身份伪装**。这行不产生汇编指令，它是给编译器看的。它说：“把这个 `shellcode` 数组当成一个函数入口来看待。”
    * 这是 C 语言最强大的地方：**模糊了数据和代码的边界**。

```c
execute_shellcode();
```
* **讲解：** **扣动扳机**。CPU 的指令指针（RIP）直接跳进了你的 `shellcode` 数组。
    * 此时，数组里的 `0x48`, `0xc7` 等不再是数字，而是变成了 `mov`, `syscall` 等指令。

```c
printf("\n[+] Execution finished safely. Returning to normal flow.\n");
```
* **讲解：** **完美收官**。证明我们在执行完机器码后，还能安全地回到正常的 C 程序逻辑，没有把整个系统跑飞（这归功于我们 shellcode 末尾那个 `0xc3`，即 `ret` 指令）。

In [1]:
!gcc shellcode_hnu.c -o shellcode_hnu
!./shellcode_hnu

[*] System Hacker Mode Activated
[*] Shellcode address: 0x6490db79e020
[*] Page start address: 0x6490db79e000
[+] NX Bit bypassed successfully using mprotect!
[+] Executing data array as machine instructions...

I Love Hunan University!

[+] Execution finished safely.
